In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Spark Tables")
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

print("Spark Session Created")
print("Catalog implementation:", spark.conf.get("spark.sql.catalogImplementation"))

Spark Session Created
Catalog implementation: hive


In [5]:

volume_path = "/home/jovyan/work/data/processed/orders_parquet"

print("\nVerifying Spark Read:")
verified_orders = spark.read.parquet(volume_path)
verified_orders.printSchema()
verified_orders.show(5, truncate=False)


Verifying Spark Read:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_timestamp: timestamp_ntz (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|order_id|customer_id|product_id|quantity|price|order_timestamp    |total_amount|order_date|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|3       |101        |P102      |3       |100  |2026-01-02 09:00:00|300         |2026-01-02|
|4       |103        |P103      |2       |700  |2026-01-02 12:00:00|1400        |2026-01-02|
|1       |101        |P100      |2       |500  |2026-01-01 10:00:00|1000        |2026-01-01|
|2       |102        |P101      |1       |200  |2026-01-01 11:00:00|

## Create Hive schema, beore table creation

In [6]:
spark.sql("DROP DATABASE IF EXISTS ecommerce CASCADE")
spark.sql("CREATE DATABASE ecommerce LOCATION '/home/jovyan/work/warehouse/ecommerce.db'")

DataFrame[]

### Create Hive Table under the new schema

In [9]:
spark.sql("USE ecommerce")
spark.sql("DROP TABLE IF EXISTS ecommerce.orders")
spark.sql("""
CREATE TABLE ecommerce.orders (
    order_id INT,
    customer_id INT,
    product_id STRING,
    quantity INT,
    price INT,
    order_timestamp TIMESTAMP,
    total_amount INT,
    order_date DATE
) USING PARQUET
LOCATION '/home/jovyan/work/data/processed/orders_parquet'
""")

DataFrame[]

In [18]:
query = "SELECT * FROM ecommerce.orders  where total_amount > 300 ORDER BY order_id"
result = spark.sql(query)
result.show()


+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|order_id|customer_id|product_id|quantity|price|    order_timestamp|total_amount|order_date|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+
|       1|        101|      P100|       2|  500|2026-01-01 10:00:00|        1000|2026-01-01|
|       4|        103|      P103|       2|  700|2026-01-02 12:00:00|        1400|2026-01-02|
+--------+-----------+----------+--------+-----+-------------------+------------+----------+

